# Figure Generation
Generates all main paper figures from the seismic results CSV.
Figures: dnu vs numax, logg comparison, radius comparison, mass by cluster,
star vs cluster age, age boxplot, age residuals, and appendix diagnostic plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import minimize
import os

fsr = pd.read_csv('../data/seismic_results.csv')
ages = pd.read_csv('../data/age_results.csv')
outdir = '../../Downloads/revision_1'  # adjust as needed

print(f'Loaded {len(fsr)} targets, {len(ages)} age entries')

## Fig 8: Corrected dnu vs numax

In [ ]:
cluster_colors = {'Casado_Alessi_1':'#e41a1c','NGC_752':'#377eb8','Theia_6046':'#4daf4a',
    'Theia_844':'#984ea3','HSC_95':'#a65628','LISC_3534':'#f781bf',
    'Theia_1188':'#66c2a5','Theia_1297':'#fc8d62','Unknown_3':'#8da0cb'}
cluster_markers = {'Casado_Alessi_1':'o','NGC_752':'s','Theia_6046':'^',
    'Theia_844':'D','HSC_95':'p','LISC_3534':'*',
    'Theia_1188':'<','Theia_1297':'>','Unknown_3':'X'}

fig, ax = plt.subplots(figsize=(7,6))
for cl in sorted(fsr['cluster'].unique()):
    sub = fsr[fsr['cluster']==cl]
    ax.errorbar(sub['numax_corr'], sub['dnu_corr'],
                xerr=sub['numax_err_up'], yerr=sub['dnu_err_up'],
                fmt=cluster_markers.get(cl,'o'), color=cluster_colors.get(cl,'gray'),
                markersize=6, ecolor='gray', elinewidth=0.5, capsize=2,
                label=cl.replace('_',' '))
nu = np.linspace(5,200,200)
ax.plot(nu, 0.263*nu**0.772, 'k--', lw=1, alpha=0.5, label='scaling')
ax.set_xlabel('numax (uHz)'); ax.set_ylabel('dnu (uHz)')
ax.legend(fontsize=7, ncol=2, loc='upper left')
plt.tight_layout()
plt.savefig(f'{outdir}/fig8_dnu_vs_numax.jpeg', dpi=200, bbox_inches='tight')
plt.show()

## Fig 9: Log g two panel

In [ ]:
def rb_cal(logg):
    return logg + 0.4496 - 0.0036*logg - 0.0224*logg**2

multi_clusters = ['Casado_Alessi_1','NGC_752','Theia_6046','Theia_844']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5))
valid = fsr.dropna(subset=['logg_gspspec','logg_seis'])
is_multi = valid['cluster'].isin(multi_clusters)

for ax, use_cal, title in [(ax1, False, 'Raw GSP-Spec log g'),
                             (ax2, True, 'Calibrated (Recio-Blanco+23)')]:
    lc = rb_cal(valid['logg_gspspec']) if use_cal else valid['logg_gspspec']
    m = is_multi; s = ~is_multi
    ax.errorbar(valid.loc[m,'logg_seis'], lc[m], xerr=valid.loc[m,'logg_err_up'],
                yerr=0.15, fmt='o', color='#8B7355', markersize=6,
                ecolor='#8B7355', elinewidth=0.8, capsize=2, label='Multi-star cluster')
    ax.errorbar(valid.loc[s,'logg_seis'], lc[s], xerr=valid.loc[s,'logg_err_up'],
                yerr=0.15, fmt='o', color='#2AA198', markersize=6,
                ecolor='#2AA198', elinewidth=0.8, capsize=2, label='Single star')
    ax.plot([1.5,3.5],[1.5,3.5],'k--',lw=1)
    if not use_cal:
        xr = np.linspace(1.5,3.2,100)
        ax.plot(xr, rb_cal(xr), 'r--', lw=1.5, label='Recio-Blanco+2023')
    ax.set_xlim(1.5,3.2); ax.set_ylim(1.5,3.2)
    ax.set_xlabel('Seismic log g (dex)'); ax.set_ylabel('Catalog log g (dex)')
    ax.set_title(title)
    d = lc.values - valid['logg_seis'].values
    ax.text(0.05, 0.95, f'N={len(d)}\nBias={np.mean(d):.2f}\nRMSE={np.sqrt(np.mean(d**2)):.2f}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(f'{outdir}/fig9_logg_twopanel.jpeg', dpi=200, bbox_inches='tight')
plt.show()

## Fig 10: Radius comparison
## Fig 11: Mass by cluster
## Age plots (star vs cluster, boxplot, residuals)

See the full figure generation in scripts/run_pysyd_plots.py or regenerate from the seismic_results.csv using the patterns above.

In [ ]:
# Radius
fig, ax = plt.subplots(figsize=(7,7))
vr = fsr.dropna(subset=['radius_flame','R_seis'])
imr = vr['cluster'].isin(multi_clusters)
ax.errorbar(vr.loc[imr,'radius_flame'], vr.loc[imr,'R_seis'],
            xerr=[vr.loc[imr,'radius_flame']-vr.loc[imr,'radius_flame_lower'],
                  vr.loc[imr,'radius_flame_upper']-vr.loc[imr,'radius_flame']],
            yerr=vr.loc[imr,'R_err_up'], fmt='o', color='#8B7355', markersize=6,
            ecolor='#8B7355', elinewidth=0.8, capsize=2, label='Multi-star cluster')
s3 = ~imr
if s3.sum() > 0:
    ax.errorbar(vr.loc[s3,'radius_flame'], vr.loc[s3,'R_seis'],
                xerr=[vr.loc[s3,'radius_flame']-vr.loc[s3,'radius_flame_lower'],
                      vr.loc[s3,'radius_flame_upper']-vr.loc[s3,'radius_flame']],
                yerr=vr.loc[s3,'R_err_up'], fmt='o', color='#2AA198', markersize=6,
                ecolor='#2AA198', elinewidth=0.8, capsize=2, label='Single star')
lm = [0, max(vr['R_seis'].max(), vr['radius_flame'].max())*1.15]
ax.plot(lm, lm, 'k--', lw=1)
ax.set_xlabel('Catalog Radius (Rsun)'); ax.set_ylabel('Seismic Radius (Rsun)')
ax.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig(f'{outdir}/fig10_radius.jpeg', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Mass by cluster
fig, ax = plt.subplots(figsize=(10,6))
pcl = ['Casado_Alessi_1','NGC_752','Theia_6046','Theia_844']
cmass = {'Casado_Alessi_1':'#4C9BD6','NGC_752':'#E8893C','Theia_6046':'#2AA198','Theia_844':'#9B59B6'}
for i, cl in enumerate(pcl):
    sub = fsr[fsr['cluster']==cl]; color = cmass[cl]
    xj = np.random.default_rng(42).uniform(-0.15, 0.15, len(sub))
    ax.errorbar(i+xj, sub['M_seis'], yerr=sub['M_err_up'],
                fmt='o', color=color, markersize=8, ecolor=color, elinewidth=1.2, capsize=3)
    ax.plot([i-0.3,i+0.3], [sub['M_seis'].median()]*2, color=color, linewidth=3)
ax.set_xticks(range(len(pcl)))
ax.set_xticklabels([c.replace('_','\n') for c in pcl], fontsize=11)
ax.set_xlabel('Cluster'); ax.set_ylabel('Seismic Mass (Msun)'); ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig(f'{outdir}/fig11_mass.jpeg', dpi=200, bbox_inches='tight')
plt.show()